# Apriori

## Importar librerías

In [1]:
!pip install apyori

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Data Preprocessing

In [3]:
dataset = pd.read_csv('Market_Basket_Optimisation.csv', header = None) #header = None para que no confunda la primera fila con el título de las columnas
transactions = []
for i in range(0, 7501):
  transactions.append([str(dataset.values[i,j]) for j in range(0, 20)]) #creo una lista de listas

## Entrenar el algoritmo de apriori Training

In [7]:
from apyori import apriori
rules = apriori(transactions = transactions, min_support = 0.003, min_confidence = 0.2, min_lift = 3, min_length = 2)
#min_support = 0.003 porque queremnos items que salgan al menos 3 veces al día-> 3*7/7.500 y redondeamos

## Visualización de los resultados

### Displaying the first results coming directly from the output of the apriori function

In [8]:
results = list(rules)

In [20]:
results[0] #pimera norma, para facilitar la lectura

RelationRecord(items=frozenset({'chicken', 'light cream'}), support=0.004532728969470737, ordered_statistics=[OrderedStatistic(items_base=frozenset({'light cream'}), items_add=frozenset({'chicken'}), confidence=0.29059829059829057, lift=4.84395061728395)])

### Putting the results well organised into a Pandas DataFrame

In [18]:
def inspect(results):
    lhs         = [tuple(result[2][0][0])[0] for result in results]
    rhs         = [tuple(result[2][0][1])[0] for result in results]
    supports    = [result[1] for result in results]
    confidences = [result[2][0][2] for result in results]
    lifts       = [result[2][0][3] for result in results]
    return list(zip(lhs, rhs, supports, confidences, lifts))
resultsinDataFrame = pd.DataFrame(inspect(results), columns = ['Left Hand Side', 'Right Hand Side', 'Support', 'Confidence', 'Lift'])

El código simplemente recorre todos los resultados y extrae esas cuatro partes clave para meterlas en un DataFrame legible.
🧠 lhs = [tuple(result[2][0][0])[0] for result in results]

result[2] → lista de estadísticas (OrderedStatistic).

[0] → coge la primera regla (la más fuerte o principal).

[0] otra vez → coge el antecedente (items_base), que viene como frozenset.

tuple(...)[0] → convierte ese frozenset en una tupla y saca el primer elemento (porque normalmente hay solo uno).

➡️ En resumen: obtiene el producto del lado izquierdo de la regla (Left Hand Side).
Ejemplo: "leche".

🧠 rhs = [tuple(result[2][0][1])[0] for result in results]

Igual que la anterior, pero para el lado derecho de la regla (items_add).
Ejemplo: "pan".

supports = [result[1] for result in results]

result[1] es directamente el soporte de la combinación (A ∩ B).

🧠 confidences = [result[2][0][2] for result in results]

result[2][0][2] → dentro del primer OrderedStatistic, posición 2 = confianza.

### Displaying the results non sorted

In [19]:
resultsinDataFrame

,Left Hand Side,Right Hand Side,Support,Confidence,Lift
0,light cream,chicken,0.004533,0.290598,4.843951
1,mushroom cream sauce,escalope,0.005733,0.300699,3.790833
2,pasta,escalope,0.005866,0.372881,4.700812
3,fromage blanc,honey,0.003333,0.245098,5.164271
4,herb & pepper,ground beef,0.015998,0.323450,3.291994
...,...,...,...,...,...
155,olive oil,spaghetti,0.003066,0.216981,3.632981
156,ground beef,spaghetti,0.003066,0.211009,3.532991
157,tomatoes,spaghetti,0.003066,0.261364,4.376091
158,spaghetti,olive oil,0.003333,0.211864,3.223519


### Displaying the results sorted by descending lifts

In [21]:
resultsinDataFrame.nlargest(n = 10, columns = 'Lift')

,Left Hand Side,Right Hand Side,Support,Confidence,Lift
97,frozen vegetables,mineral water,0.003066,0.383333,7.987176
150,frozen vegetables,mineral water,0.003066,0.383333,7.987176
96,olive oil,mineral water,0.003333,0.294118,6.128268
149,olive oil,mineral water,0.003333,0.294118,6.128268
132,mineral water,olive oil,0.003866,0.402778,6.128268
59,mineral water,olive oil,0.003866,0.402778,6.115863
50,tomato sauce,spaghetti,0.003066,0.216981,5.535971
122,tomato sauce,spaghetti,0.003066,0.216981,5.535971
28,fromage blanc,nan,0.003333,0.245098,5.178818
3,fromage blanc,honey,0.003333,0.245098,5.164271


# Otra forma más fácil de visualizarlo

In [23]:
pip install mlxtend


   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 9.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [25]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [35]:
## 1️⃣ Convertir tus transacciones en formato binario (True/False)
te = TransactionEncoder()
df = pd.DataFrame(te.fit_transform(transactions),
                  columns=te.columns_)

El algoritmo Apriori (y la función association_rules) no puede trabajar con listas de texto directamente.
Necesita un DataFrame numérico o booleano donde:

Cada columna sea un producto.

Cada fila sea una transacción.

Los valores sean True/False (o 1/0) según si el producto está o no está en esa transacción


In [36]:
# 2️⃣ Ejecutar Apriori con tus mismos parámetros del curso
frequent_items = apriori(df, min_support=0.003, use_colnames=True)
#use_colnames=True -> Usa los nombres reales de los productos como etiquetas de los itemsets


In [32]:
# 3️⃣ Generar las reglas automáticamente (sin inspect)
rules = association_rules(frequent_items, metric="confidence", min_threshold=0.2)
rules = rules[rules['lift'] >= 3]  # equivalente a min_lift = 3
#Toma los itemsets frecuentes que encontraste con apriori() y calcula todas las posibles reglas A → B que se pueden formar con ellos.

In [43]:
# 4️⃣ Mostrar las reglas limpias

rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']] \
    .sort_values(by='lift', ascending=False) \
    .head(20).reset_index(drop=True)


,antecedents,consequents,support,confidence,lift
0,"(frozen vegetables, soup)","(mineral water, milk)",0.003066,0.383333,7.987176
1,"(frozen vegetables, soup)","(mineral water, nan, milk)",0.003066,0.383333,7.987176
2,"(frozen vegetables, soup, nan)","(mineral water, milk)",0.003066,0.383333,7.987176
3,"(olive oil, frozen vegetables)","(mineral water, nan, milk)",0.003333,0.294118,6.128268
4,"(olive oil, frozen vegetables, nan)","(mineral water, milk)",0.003333,0.294118,6.128268
5,"(olive oil, frozen vegetables)","(mineral water, milk)",0.003333,0.294118,6.128268
6,"(mineral water, whole wheat pasta)","(olive oil, nan)",0.003866,0.402778,6.128268
7,"(mineral water, nan, whole wheat pasta)",(olive oil),0.003866,0.402778,6.115863
8,"(mineral water, whole wheat pasta)",(olive oil),0.003866,0.402778,6.115863
9,"(soup, nan, milk)","(frozen vegetables, mineral water)",0.003066,0.201754,5.646864
